<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 4: Regression for Tool Wear Estimation and Prediction</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week04/lab04_tool_wear_regression.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University**

**Graded individual Lab · Approximately 90 minutes**
**Version:** Student Notebook

Open in Colab and save a personal copy before editing. Canvas contains the official submission requirements and due date.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Measurements to wear estimation

![Concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/experiment_overview.png)

Start with the prepared CSV: each cut becomes one row of sensor features and measured wear. The small plot illustrates one input; all Lab models use 11 sensor features. Cut number is not an input.

## Overview

Use 11 sensor features to estimate tool wear and compare Linear, Ridge,
degree-2 Polynomial regression, and RBF SVR.

By the end, you can:
- prepare inputs and scale them using training data;
- fit models and compare parameter settings using validation RMSE;
- interpret prediction errors on later cuts.

**Your work:** four fit/predict edits in CP2 and four short checkpoint responses.
Run **PROVIDED** cells unchanged; use their comments when you need help.
Unfinished tasks print reminders and skip dependent results.

**Time:** preparation/example 40 min; model experiments 30 min; evaluation/submission 20 min.
**10 points:** CP1 2, CP2 4, CP3 2, CP4 2. Optional work is ungraded.
Credit is based on completion and evidence-based effort, not a target accuracy.

## Software and data

Use NumPy, pandas, Matplotlib, and scikit-learn in Colab. Locally, install these
packages and Jupyter in your Python environment. The loader retrieves the course
CSV online; for offline use, put `phm2010_features.csv` beside this notebook.

The **PHM Society 2010 Data Challenge** provides wear labels for c1, c4, and c6.
Our table has 945 rows: 315 cuts per cutter. The course-derived target
`wear_mean_um` is the mean of three flute-wear measurements, in µm.
Features summarize full recordings of force (N), vibration (g), and AE RMS (V),
sampled at 50 kHz. No raw-signal processing is required here.

This is a course-specific task, not the original competition benchmark.
See the [feature README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md)
for provenance and dataset-rights limitations, and the
[data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
for feature definitions.

In [ ]:
# [PROVIDED SETUP] Import tools; importing does not fit a model.
# Path handles file locations so the loader can look for a local CSV.
from pathlib import Path
# NumPy (np) supplies array operations and square roots for RMSE.
import numpy as np
# pandas (pd) reads CSV files and organizes rows and columns into tables.
import pandas as pd
# Matplotlib (plt) creates the figures; display shows formatted tables.
import matplotlib.pyplot as plt
from IPython.display import display
# scikit-learn supplies model classes. Importing a class only makes it available.
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
# StandardScaler rescales inputs; PolynomialFeatures creates squares and products.
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
# These functions compare measured wear with predictions to calculate errors.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Readable labels are used in all supplied figures.
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})


In [ ]:
# [PROVIDED SETUP] Prefer a local CSV; otherwise read the existing course file.
# Path represents a file location. No source data are modified.
data_path = Path('phm2010_features.csv')
# exists() asks whether the file is present; not reverses True/False.
if not data_path.exists():
    # Try the relative path used when the notebook stays inside the course repository.
    data_path = Path('../../data/phm2010/features/phm2010_features.csv')
# Read the local file when found; otherwise the else branch downloads the same CSV.
if data_path.exists():
    features = pd.read_csv(data_path)
else:
    features = pd.read_csv('https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv')

# Each table below contains one cutter, ordered from earlier to later cuts.
# The comparison creates True/False row flags. .loc keeps matching rows; sort_values orders
# them.
c1 = features.loc[features['cutter_id'] == 'c1'].sort_values('cut_number')
c4 = features.loc[features['cutter_id'] == 'c4'].sort_values('cut_number')
c6 = features.loc[features['cutter_id'] == 'c6'].sort_values('cut_number')
print('Rows for c1, c4, c6:', len(c1), len(c4), len(c6))
# Double brackets select several columns; head() displays only the first five rows.
display(c1[['cutter_id', 'cut_number', 'wear_mean_um']].head())


## A. Prepare the data

Split **each cutter chronologically**, then pool matching partitions.

| Partition | Cuts per cutter | Pooled rows | Purpose |
|---|---|---|---|
| Train ≈50% | 1–157 | 471 | Fit scaler and models |
| Validation ≈20% | 158–220 | 189 | Select model and settings |
| Test ≈30% | 221–315 | 285 | Evaluate the locked choice |

`.iloc[start:stop]` excludes the stop position: `157:220` selects 63 rows.
The integer boundaries approximate 50/20/30; do not shuffle the cuts.

![Concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/training_workflow.png)

Split each cutter at cuts 157/158 and 220/221. Select using validation RMSE; the full trajectories provide context, not a basis for tuning on test data.

In [ ]:
# [PROVIDED] Keep earlier, middle and later cuts separate within each cutter.
# pd.concat stacks tables vertically; ignore_index gives new row labels.
# Row positions start at 0; :157 keeps positions 0–156 (cuts 1–157) from EACH cutter.
train = pd.concat([c1.iloc[:157], c4.iloc[:157], c6.iloc[:157]], ignore_index=True)
# 157:220 keeps positions 157–219 (cuts 158–220); the stop position is excluded.
valid = pd.concat([c1.iloc[157:220], c4.iloc[157:220], c6.iloc[157:220]], ignore_index=True)
# 220: keeps all remaining rows (cuts 221–315) for the final evaluation.
test = pd.concat([c1.iloc[220:], c4.iloc[220:], c6.iloc[220:]], ignore_index=True)
print('Train / validation / test rows:', len(train), len(valid), len(test))


# The figure shows partition boundaries only, not held-out target values.
# fig is the whole figure; ax is the plotting area where bars and labels are drawn.
fig, ax = plt.subplots(figsize=(9, 3.3))
# barh draws horizontal bars. Their widths are cut counts; left sets the starting edge.
ax.barh(['c1', 'c4', 'c6'], [157, 157, 157], left=0.5, color='#086759', label='Train')
ax.barh(['c1', 'c4', 'c6'], [63, 63, 63], left=157.5, color='#C08923', label='Validation')
ax.barh(['c1', 'c4', 'c6'], [95, 95, 95], left=220.5, color='#2468A0', label='Test')
ax.set(xlabel='Cut number (sequence)', ylabel='Cutter', title='Same chronological split within each cutter')
# The legend identifies colors; bbox_to_anchor places it below the plotting area.
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
# Adjust spacing so labels fit; show() displays the completed figure.
fig.tight_layout()
plt.show()


### Prepare X and y

One row is one cut. `X_train` contains 471 rows × 11 sensor features;
`y_train` contains the corresponding 471 wear values in the same order.
A list of column names gives a 2D input table; one target name gives a 1D series.
Cut number, cutter ID, and other wear columns are excluded from X.

### Notation: a number, a vector, and a table

| Symbol | Meaning |
|---|---|
| $x_{ij}$ | One scalar: feature $j$ for cut $i$ |
| $\mathbf{x}_i$ | The 11 features of one cut, a column vector |
| $\mathbf{X}$ | All cuts in a matrix: rows = cuts, columns = features |
| $y_i$, $\hat y_i$ | One measured wear and one predicted wear, in µm |
| $e_i=y_i-\hat y_i$ | One residual; positive means underprediction |
| $\mathbf y$, $\hat{\mathbf y}$, $\mathbf e$ | Vectors of measured wear, predictions, and residuals |

In math, vectors use bold lowercase and matrices use bold uppercase.
`X_train` is a 471 × 11 table; `y_train` is stored as a 1D sequence with
shape `(471,)` in Python. Scaled inputs are $z_{ij}$ and $\mathbf z_i$.
A single-feature illustration uses scalar $x_i$; the real Lab uses 11 inputs.

![Concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/sensor_inputs_target.png)

$\mathbf{x}_i$: 11 sensor inputs for one cut. $y_i$: measured mean flute wear; $\hat y_i$: predicted wear; $e_i=y_i-\hat y_i$: residual. Wear remains in µm.

In [ ]:
# [PROVIDED] Fixed sensor inputs; do not add/remove columns for this Lab.
default_features = [
    'force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
    'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
    'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd',
]
# Select the 11 sensor columns for training; keep all rows and their order.
X_train = train[default_features]   # Sensor features: forces N, vibrations g, AE-RMS V.
# Use exactly the same columns for validation so each feature keeps its meaning.
X_valid = valid[default_features]   # Same columns, in the same order.
# Select the measured training target as a 1D series; one value per training cut.
y_train = train['wear_mean_um']     # Measured mean flute wear, in micrometers.
# Keep measured validation wear for scoring later; it is not an input to predict.
y_valid = valid['wear_mean_um']
# shape reports (rows, columns) for X and (rows,) for the one-dimensional y.
print('Training X shape:', X_train.shape)
print('Training y shape:', y_train.shape)
print('Training wear range (µm):', round(y_train.min(), 2), 'to', round(y_train.max(), 2))
# Test X and y are deliberately not used during candidate selection.


## [REQUIRED CHECKPOINT 1] Describe the regression task

State what one row, X and y represent, with the target unit. Record pooled
train/validation/test counts and explain why we keep cut order within each cutter.

**Your response:** TODO: Write 2–3 sentences.

### Scale the inputs

Standardization makes differently scaled sensor inputs comparable:

$$z_{ij}=\frac{x_{ij}-\mu_{j,\mathrm{train}}}{s_{j,\mathrm{train}}}.$$

`fit_transform` learns each training column's mean and population standard
deviation, then scales it. `transform` applies those same statistics to validation.
Scaled inputs are dimensionless; **y stays in µm**. Scaling does not remove features.

In [ ]:
# [GUIDED — RUN UNCHANGED] Fit one scaler for candidate comparison.
# Create an unfitted scaler; it has not seen any sensor values yet.
scaler = StandardScaler()
# Learn 11 training means/spreads and scale the 471 training rows in one step.
X_train_scaled = scaler.fit_transform(X_train)
# Use the stored training statistics on 189 validation rows; do not learn new statistics.
X_valid_scaled = scaler.transform(X_valid)
print('Scaled training shape:', X_train_scaled.shape)
# A scaled value of 2 means two training standard deviations above the mean.


## B. Guided example — Linear regression

A single-feature illustration is $\hat y_i=\beta_0+\beta_1x_i$.
With our 11 standardized sensor inputs:

$$\hat y_i=\beta_0+\sum_{j=1}^{11}\beta_j z_{ij}
=\beta_0+\boldsymbol\beta^{\mathsf T}\mathbf z_i.$$

The coefficients and intercept are learned by `fit`; `predict` uses
them with new inputs and does not receive the validation answers.
The hat means predicted. There is no residual term added to the prediction;
the observed value satisfies $y_i=\hat y_i+e_i$ by definition.
Coefficient size alone is not feature importance when inputs are correlated.

In [ ]:
# [GUIDED EXAMPLE] Read this pattern before the two student model cells.
linear_model = LinearRegression()  # Create an unfitted model with an intercept.
linear_model.fit(X_train_scaled, y_train)  # Learn from training inputs AND targets.
linear_valid_pred = linear_model.predict(X_valid_scaled)  # Predict without y_valid.
linear_train_pred = linear_model.predict(X_train_scaled)  # Check fit on known rows.

# Store each quantity explicitly; RMSE is the square root of mean squared error.
# Compare each measured value with its matching prediction; average absolute errors (µm).
linear_valid_mae = mean_absolute_error(y_valid, linear_valid_pred)
# Square errors, average them, then take the square root to return to µm.
linear_valid_rmse = np.sqrt(mean_squared_error(y_valid, linear_valid_pred))
# R² compares squared error with the validation-mean reference; it is unitless.
linear_valid_r2 = r2_score(y_valid, linear_valid_pred)
# Score known training rows separately to compare training fit with validation performance.
linear_train_rmse = np.sqrt(mean_squared_error(y_train, linear_train_pred))
print('Training RMSE (µm):', round(linear_train_rmse, 2))
print('Validation MAE (µm):', round(linear_valid_mae, 2))
print('Validation RMSE (µm):', round(linear_valid_rmse, 2))
print('Validation R² (unitless):', round(linear_valid_r2, 3))


### How to read the numbers and residuals

For measured wear $y_i$, prediction $\hat y_i$, and $n$ evaluated cuts:

$$e_i=y_i-\hat y_i,\quad
\mathrm{MAE}=\frac{1}{n}\sum_i|e_i|,\quad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i e_i^2}.$$

MAE is mean absolute error; RMSE gives large errors more weight. Both are in
**µm**, and smaller is better. A residual above zero means underestimation.

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

Here $\bar y$ is the mean of the **evaluated partition**. R² is unitless,
equals 1 for perfect predictions, and can be negative. A negative value means
larger squared error than predicting that partition's mean. That reference
mean is not a deployable prediction learned from training.

The supplied baseline below instead predicts the **training mean** everywhere.
It provides a simple reference, not a fifth candidate in this exercise.

In [ ]:
# [PROVIDED] A baseline learned from training only.
# np.full creates one prediction per validation row, all equal to the TRAINING wear mean.
baseline_valid_pred = np.full(len(y_valid), y_train.mean())
baseline_valid_rmse = np.sqrt(mean_squared_error(y_valid, baseline_valid_pred))
print('Training-mean baseline validation RMSE (µm):', round(baseline_valid_rmse, 2))

# [PROVIDED PLOT] Display one cutter to explain the visual before the comparison.
# A mask is True for c1 rows and False elsewhere; use it on targets AND predictions.
c1_valid_mask = valid['cutter_id'] == 'c1'
# Create two side-by-side panels: axes[0] is left and axes[1] is right.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
# .loc[mask, column] retrieves cut numbers for the same c1 rows shown on the y-axis.
axes[0].plot(valid.loc[c1_valid_mask, 'cut_number'], y_valid[c1_valid_mask], label='Measured')
axes[0].plot(valid.loc[c1_valid_mask, 'cut_number'], linear_valid_pred[c1_valid_mask], label='Predicted')
axes[0].set(title='Linear regression: c1 validation', xlabel='Cut number', ylabel='Mean flute wear (µm)')
axes[0].legend()
# Subtract in matching row order: measured minus predicted is the residual in µm.
axes[1].plot(valid.loc[c1_valid_mask, 'cut_number'], y_valid[c1_valid_mask] - linear_valid_pred[c1_valid_mask])
# The horizontal zero line marks perfect agreement, not another fitted model.
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(title='Measured minus predicted', xlabel='Cut number', ylabel='Residual (µm)')
fig.tight_layout()
plt.show()


## [REQUIRED CHECKPOINT 2] Build models and compare settings

Follow the Linear example: complete one `fit` and one `predict` call for
Ridge and SVR (**four edits**). `None` marks unfinished work. Keep the initial
settings at **alpha = 1** and **C = 100**, then inspect their feedback.
Polynomial regression and the parameter-comparison loops are provided.

### Ridge regression

$$\min_{\beta_0,\boldsymbol\beta}\sum_{i\in\mathrm{train}}(y_i-\hat y_i)^2
+\alpha\sum_{j=1}^{11}\beta_j^2.$$

Larger alpha penalizes large coefficients more strongly; the intercept is not
penalized. This is not feature selection and need not improve validation error.
The experiment compares **alpha = 0.1, 1, 10**.

![Concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/regression_families.png)

Synthetic illustration, not PHM results. Ridge stays linear; Polynomial can curve. Greater flexibility need not improve validation accuracy.

In [ ]:
# [STUDENT TASK] Complete the two TODO expressions; keep other lines unchanged.
# Step 1 — Create the model object that will remember the fitted relationship.
# alpha controls the penalty on large coefficients; 1.0 is our starting value.
# Creating this object does not learn coefficients yet.
ridge_model = Ridge(alpha=1.0)

# Step 2 — Learn from the training data, following the completed Linear example.
# X_train_scaled: 471 cuts × 11 standardized sensor features (unitless).
# y_train: the 471 matching measured wear values (µm), in the same row order.
# Replace the TODO comment with a fit call on the model created above.
# Fitting updates that model object; do not use validation or test targets here.
# TODO 1: Fit ridge_model using X_train_scaled and y_train.

# Step 3 — Apply the fitted relationship to the validation inputs.
# X_valid_scaled: 189 cuts × the same 11 features, scaled with TRAINING statistics.
# Replace None with a prediction call on your fitted model. Do not pass y_valid.
# The result is a 1D array of 189 predicted wear values (µm), one per input row.
# The equals sign stores these predictions under the variable name on the left.
ridge_valid_pred = None  # TODO 2: Predict using X_valid_scaled.
# None is only an unfinished placeholder; downstream cells wait until it is replaced.

In [ ]:
# [PROVIDED FEEDBACK] Keep the initial student result separate from later tuning.
# Keep a reference to this initial model; later tuning creates different model objects.
ridge_initial_model = ridge_model
# Keep these initial predictions so feedback is not replaced by the tuned predictions.
ridge_initial_pred = ridge_valid_pred
# Skip scoring while the student prediction is still None (unfinished).
if ridge_initial_pred is not None:
    # A valid prediction has one finite numeric value for each validation cut.
    # Convert predictions to a NumPy array without changing their row order or wear units.
    ridge_initial_pred = np.asarray(ridge_initial_pred)
    ridge_initial_rmse = np.sqrt(mean_squared_error(y_valid, ridge_initial_pred))
    print('Initial Ridge validation RMSE (µm):', round(ridge_initial_rmse, 3))
    print('This is your initial model, before the supplied parameter comparison.')
else:
    print('Complete the two Ridge expressions above to see initial feedback.')


### Candidate 3: second-degree Polynomial regression

A two-input illustration is $\hat y_i=\beta_0+\beta_1z_{i1}+\beta_2z_{i2}+\beta_3z_{i1}^2+\beta_4z_{i1}z_{i2}+\beta_5z_{i2}^2$.
Squares allow curvature; a product allows one input's effect to depend on
another input. The equation is nonlinear in the original inputs but linear
in its coefficients, so we still use `LinearRegression` after expansion.

Our 11 inputs become **77 columns**: 11 originals, 11 squares and 55 pairwise
products. More flexibility does not guarantee better later-cut predictions.
`include_bias=False` avoids a duplicate constant because the regression
already has an intercept. Run the transformation code unchanged.

In [ ]:
# [PROVIDED] Expand the training and validation columns in exactly the same way.
# Degree 2 adds squares and pairwise products. No constant column: the model supplies its
# intercept.
polynomial = PolynomialFeatures(degree=2, include_bias=False)
# Expand 11 scaled inputs to 77 columns: 11 originals + 11 squares + 55 products.
X_train_poly = polynomial.fit_transform(X_train_scaled)
# Apply the same column expansion and order to validation; do not change the target.
X_valid_poly = polynomial.transform(X_valid_scaled)
# shape[1] is the column count; shape[0] would be the number of cuts.
print('Polynomial columns:', X_train_poly.shape[1])


In [ ]:
# [PROVIDED COMPARISON] Degree stays fixed; this is not a student code task.
# The inputs now include nonlinear terms, but the fitted coefficients still enter linearly.
polynomial_model = LinearRegression()
# Learn coefficients from EXPANDED training inputs and the original wear targets.
polynomial_model.fit(X_train_poly, y_train)
# Predict using the matching 77 validation columns; outputs remain wear in µm.
polynomial_valid_pred = polynomial_model.predict(X_valid_poly)


### Support Vector Regression (SVR): an epsilon-insensitive loss

RBF SVR uses similarity between scaled input rows to represent a nonlinear
relation. Use the original **11 scaled columns**, not polynomial features.

$$L_{\epsilon_{\mathrm{SVR}}}(e_i)
=\max(0,\,|e_i|-\epsilon_{\mathrm{SVR}}).$$

| Setting | Meaning / our choice |
|---|---|
| `epsilon=1.0` | Zero loss within ±1 µm of the prediction; fixed here. This is not a confidence interval or guaranteed accuracy. |
| `C` | Weight on loss outside the band relative to model complexity; compare **10, 100, 1000**. |
| `gamma='scale'` | Training-data rule for RBF similarity width; keep this rule fixed. |

Larger C penalizes violations more strongly, but does not guarantee better
validation performance. $e_i$ is the residual; $\epsilon_{\mathrm{SVR}}$
is a setting. Target wear remains in µm. Complete the example at C = 100.

![Concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/svr_band.png)

Synthetic illustration: the ±1 µm band has zero SVR loss; only the distance outside the band is penalized. This is not a confidence interval.

In [ ]:
# [STUDENT TASK] Complete the two TODO expressions; keep other lines unchanged.
# Step 1 — Create the model object that will remember the fitted relationship.
# kernel='rbf' allows a nonlinear relation based on similarity between input rows.
# C=100.0 is the starting penalty weight; epsilon=1.0 sets a 1 µm zero-loss band.
# gamma='scale' uses a training-input rule for the similarity width.
# These settings define the model; creating it does not fit it yet.
svr_model = SVR(kernel='rbf', C=100.0, epsilon=1.0, gamma='scale')

# Step 2 — Learn from the training data, following the completed Linear example.
# X_train_scaled: 471 cuts × 11 standardized sensor features (unitless).
# y_train: the 471 matching measured wear values (µm), in the same row order.
# Replace the TODO comment with a fit call on the model created above.
# Fitting updates that model object; do not use validation or test targets here.
# TODO 3: Fit svr_model using X_train_scaled and y_train.

# Step 3 — Apply the fitted relationship to the validation inputs.
# X_valid_scaled: 189 cuts × the same 11 features, scaled with TRAINING statistics.
# Replace None with a prediction call on your fitted model. Do not pass y_valid.
# The result is a 1D array of 189 predicted wear values (µm), one per input row.
# The equals sign stores these predictions under the variable name on the left.
svr_valid_pred = None  # TODO 4: Predict using X_valid_scaled.
# None is only an unfinished placeholder; downstream cells wait until it is replaced.

In [ ]:
# [PROVIDED FEEDBACK] Keep the initial student result separate from later tuning.
# Keep a reference to this initial model; later tuning creates different model objects.
svr_initial_model = svr_model
# Keep these initial predictions so feedback is not replaced by the tuned predictions.
svr_initial_pred = svr_valid_pred
# Skip scoring while the student prediction is still None (unfinished).
if svr_initial_pred is not None:
    # A valid prediction has one finite numeric value for each validation cut.
    # Convert predictions to a NumPy array without changing their row order or wear units.
    svr_initial_pred = np.asarray(svr_initial_pred)
    svr_initial_rmse = np.sqrt(mean_squared_error(y_valid, svr_initial_pred))
    print('Initial Svr validation RMSE (µm):', round(svr_initial_rmse, 3))
    print('This is your initial model, before the supplied parameter comparison.')
else:
    print('Complete the two Svr expressions above to see initial feedback.')


### Checkpoint 2 response — before running the experiment

Predict how increasing alpha or C might change training and validation
RMSE. One sentence is enough; a reasonable hypothesis need not be correct.
After running Experiments A/B and the tables/plots below, return here and
add one sentence describing what you observed.

**Your response:** TODO: Hypothesis first; then one observation.

### Experiment A — Ridge

Run unchanged to compare alpha = 0.1, 1, 10. Each loop repeats the fit/predict
steps and records training and validation RMSE.

In [ ]:
# [PROVIDED EXPERIMENT] Repeat the fit/predict pattern for three settings.
# Each row uses the same training and validation observations. No test data.
# Keep these lists and the initial task-cell settings fixed for required work.
# These loops perform the required three-value parameter comparison for you.
ridge_alphas = [0.1, 1.0, 10.0]
svr_cs = [10.0, 100.0, 1000.0]
# This is True only when both student prediction placeholders have been completed.
tuning_ready = ridge_valid_pred is not None and svr_valid_pred is not None
if tuning_ready:
    # Start an empty results table; append one row after each fit.
    ridge_rows = []
    # The indented lines repeat once for each value in ridge_alphas.
    for alpha in ridge_alphas:
        # Create a fresh model for this alpha so each setting starts independently.
        candidate = Ridge(alpha=alpha)
        # Learn only from training rows; validation targets are reserved for scoring.
        candidate.fit(X_train_scaled, y_train)
        # Training predictions measure fit to data the model has already seen.
        train_prediction = candidate.predict(X_train_scaled)
        # Validation predictions measure performance on the next, unseen cuts.
        valid_prediction = candidate.predict(X_valid_scaled)
        # append adds one result row: setting, training error, then validation error.
        ridge_rows.append([alpha,
            np.sqrt(mean_squared_error(y_train, train_prediction)),
            np.sqrt(mean_squared_error(y_valid, valid_prediction))])
    # Turn the collected rows into a table; column names follow the same order as append.
    ridge_tuning = pd.DataFrame(ridge_rows,
        columns=['alpha', 'Train RMSE (µm)', 'Validation RMSE (µm)'])


### Experiment B — SVR

Run unchanged to compare C = 10, 100, 1000. Inputs, epsilon, and the gamma rule stay fixed.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
# Run this block only after both student prediction placeholders have been completed.
if tuning_ready:
    # Store SVR results separately so alpha and C are not mixed.
    svr_rows = []
    # A fresh model is created for every C; fits do not accumulate.
    for c_value in svr_cs:
        candidate = SVR(kernel='rbf', C=c_value, epsilon=1.0, gamma='scale')
        # Fit this C setting on the same training inputs and targets used for every candidate.
        candidate.fit(X_train_scaled, y_train)
        # Training predictions measure fit to data the model has already seen.
        train_prediction = candidate.predict(X_train_scaled)
        # Validation predictions measure performance on the next, unseen cuts.
        valid_prediction = candidate.predict(X_valid_scaled)
        # Save one row for this C: its value, training RMSE, and validation RMSE.
        svr_rows.append([c_value,
            np.sqrt(mean_squared_error(y_train, train_prediction)),
            np.sqrt(mean_squared_error(y_valid, valid_prediction))])
    # Create the results table after the loop finishes all three settings.
    svr_tuning = pd.DataFrame(svr_rows,
        columns=['C', 'Train RMSE (µm)', 'Validation RMSE (µm)'])


### Compare the parameter curves

Blue: training RMSE. Orange: validation RMSE. Lower is better.
Use these results to finish your CP2 observation.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if tuning_ready:
    # Round only the displayed values; selection later uses the full-precision results.
    display(ridge_tuning.round(3))
    display(svr_tuning.round(3))

    # Log axes place tenfold parameter increases at equal distances.
    # Two panels share one y-axis scale so error sizes can be compared directly.
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    # Left panel: plot each alpha against its error; marker symbols identify the two curves.
    axes[0].plot(ridge_alphas, ridge_tuning['Train RMSE (µm)'], 'o-', label='Train')
    axes[0].plot(ridge_alphas, ridge_tuning['Validation RMSE (µm)'], 's-', label='Validation')
    # Right panel: plot each C against its error using the same training/validation colors.
    axes[1].plot(svr_cs, svr_tuning['Train RMSE (µm)'], 'o-', label='Train')
    axes[1].plot(svr_cs, svr_tuning['Validation RMSE (µm)'], 's-', label='Validation')
    axes[0].set(title='Ridge', xlabel='alpha (log scale)', ylabel='RMSE (µm)')
    axes[1].set(title='RBF SVR', xlabel='C (log scale)')
    # Apply the same formatting to each panel; this loop does not fit or change any model.
    for ax in axes:
        ax.set_xscale('log')
        ax.set_ylim(bottom=0)
        # alpha here is grid transparency (0–1), unrelated to the Ridge penalty.
        ax.grid(alpha=0.2)
        ax.legend()
    fig.tight_layout()
    plt.show()


### Keep each family's best setting

Run unchanged. `idxmin()` locates the smallest validation RMSE;
the code refits Ridge and SVR with those settings.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if tuning_ready:
    # Select settings ONLY from validation RMSE; ties keep the first value.
    # Find the smallest validation RMSE row, read its alpha, and convert it to a Python number.
    best_alpha = float(ridge_tuning.loc[ridge_tuning['Validation RMSE (µm)'].idxmin(), 'alpha'])
    # Apply the same validation-only rule to the SVR table to obtain C.
    best_c = float(svr_tuning.loc[svr_tuning['Validation RMSE (µm)'].idxmin(), 'C'])
    # Replace the working model with a fresh model at the winning alpha; the initial model is
    # retained separately.
    ridge_model = Ridge(alpha=best_alpha)
    # Fit again on TRAINING only so the next family comparison uses the selected setting.
    ridge_model.fit(X_train_scaled, y_train)
    ridge_valid_pred = ridge_model.predict(X_valid_scaled)
    # Repeat for the winning C, keeping the RBF kernel, epsilon, and gamma rule unchanged.
    svr_model = SVR(kernel='rbf', C=best_c, epsilon=1.0, gamma='scale')
    svr_model.fit(X_train_scaled, y_train)
    svr_valid_pred = svr_model.predict(X_valid_scaled)
    print('Validation-selected alpha / C:', best_alpha, best_c)
else:
    print('Complete the four fit/predict edits, then restart and Run all.')
if tuning_ready:
    # Compare your preserved initial fits with the validation-selected settings.
    # Dictionary keys become column headings; each two-item list supplies Ridge and SVR values.
    initial_vs_tuned = pd.DataFrame({
        'Model': ['Ridge', 'SVR'],
        # The dot reads a model setting; str converts the number to text for the label.
        'Initial setting': ['alpha=' + str(ridge_initial_model.alpha), 'C=' + str(svr_initial_model.C)],
        'Initial validation RMSE (µm)': [ridge_initial_rmse, svr_initial_rmse],
        'Selected setting': ['alpha=' + str(best_alpha), 'C=' + str(best_c)],
        'Selected validation RMSE (µm)': [
            np.sqrt(mean_squared_error(y_valid, ridge_valid_pred)),
            np.sqrt(mean_squared_error(y_valid, svr_valid_pred))],
    })
    display(initial_vs_tuned.round(3))


## [REQUIRED CHECKPOINT 3] Select a model

Compare Linear, tuned Ridge, degree-2 Polynomial, and tuned SVR on the same
validation rows. **Select the lowest pooled validation RMSE.** Use the other
metrics and cutter results to explain limitations; do not change the rule.

### Compare four model families

Run unchanged to report overall and per-cutter metrics. A Boolean `mask`
selects one cutter's rows; the comments explain the reporting loops.

In [ ]:
# [PROVIDED REPORTING — RUN UNCHANGED] The guard prevents errors in an unfinished starter.
# and requires every listed prediction to exist before downstream reporting can run.
all_models_ready = (ridge_valid_pred is not None and polynomial_valid_pred is not None
                    and svr_valid_pred is not None)
# None means no model has been selected yet; selection fills this variable later.
selected_name = None
if all_models_ready:
    # Each list below follows the same model order.
    names = ['Linear', 'Ridge', 'Polynomial', 'SVR']
    validation_predictions = [linear_valid_pred, ridge_valid_pred, polynomial_valid_pred, svr_valid_pred]
    training_predictions = [linear_train_pred, ridge_model.predict(X_train_scaled),
                            polynomial_model.predict(X_train_poly), svr_model.predict(X_train_scaled)]
    # Collect overall metric rows here; cutter_rows will hold separate rows for each cutter.
    rows = []
    cutter_rows = []
    # range(4) gives 0, 1, 2, 3: matching positions in the names and prediction lists.
    for index in range(4):
        # Pick one model’s validation predictions without fitting it again.
        prediction = validation_predictions[index]
        # Save this model’s name, training RMSE, validation MAE, validation RMSE, and validation
        # R².
        rows.append([names[index], np.sqrt(mean_squared_error(y_train, training_predictions[index])),
                     mean_absolute_error(y_valid, prediction), np.sqrt(mean_squared_error(y_valid, prediction)),
                     r2_score(y_valid, prediction)])
        # For this model, repeat the next steps separately for c1, c4, and c6.
        for cutter in ['c1', 'c4', 'c6']:
            # True marks rows for the current cutter; applying the same mask keeps targets and
            # predictions paired.
            mask = valid['cutter_id'] == cutter
            # Calculate RMSE only on the selected cutter and store its model/cutter labels.
            cutter_rows.append([names[index], cutter, np.sqrt(mean_squared_error(y_valid[mask], prediction[mask]))])
    # Column headings match the values appended above; no new model is trained here.
    comparison = pd.DataFrame(rows, columns=['Model', 'Train RMSE (µm)', 'Validation MAE (µm)',
                                           'Validation RMSE (µm)', 'Validation R²'])
    by_cutter = pd.DataFrame(cutter_rows, columns=['Model', 'Cutter', 'Validation RMSE (µm)'])


### Read the selected model

The selected model has the smallest overall validation RMSE.
The per-cutter table shows differences hidden by the pooled result.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    display(comparison.round(3))
    # pivot rearranges results into model rows and cutter columns; the scores do not change.
    display(by_cutter.pivot(index='Model', columns='Cutter', values='Validation RMSE (µm)').round(2))
    # idxmin returns the row with minimum RMSE; equal values keep the first candidate.
    # idxmin returns a ROW LABEL, not the RMSE itself; equal minima keep the first row.
    selected_index = comparison['Validation RMSE (µm)'].idxmin()
    # Read the Model column at that winning row so the final stage knows which model to create.
    selected_name = comparison.loc[selected_index, 'Model']
    print('Selected using validation only:', selected_name)
else:
    print('Complete the four CP2 edits, then run all before selecting a model.')


### Compare validation predictions

Inspect errors across cutters. Training fit is context, not the selection score.
Negative predictions, if any, remain in the metrics without clipping.

In [ ]:
# [PROVIDED PLOT] Each family uses the same validation observations.
if all_models_ready:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    # Four x positions, one for each model; offsets separate the two bars.
    positions = np.arange(4)
    # Shift training bars left; width 0.36 leaves room for validation bars beside them.
    ax.bar(positions - 0.18, comparison['Train RMSE (µm)'], 0.36, label='Train')
    # Shift validation bars right at the same model positions for paired comparison.
    ax.bar(positions + 0.18, comparison['Validation RMSE (µm)'], 0.36, label='Validation')
    # Replace numeric positions 0–3 with the corresponding model names.
    ax.set_xticks(positions, names)
    ax.set(ylabel='RMSE (µm)', title='Model comparison after validation-based parameter selection')
    ax.legend()
    fig.tight_layout()
    plt.show()


### Checkpoint 3 response — complete before running the final test

Name the selected candidate and its chosen setting and cite its validation RMSE and one competitor's
RMSE. Use the cutter table or train–validation gap to state one caution. Explain
why you did not choose the candidate from test results.

**Your response:** TODO: Write 2–3 sentences.

## [REQUIRED CHECKPOINT 4] Refit and test

After writing CP3, lock the model and settings. Refit a **fresh scaler and model**
on the first 220 cuts per cutter (train + validation = 660 rows), then evaluate
the reserved 285 test rows. Do not retune after seeing test results.

### Refit A — prepare inputs

Run unchanged. Fit a new scaler on the 660 refit rows; only transform test inputs.

In [ ]:
# [PROVIDED FINAL EVALUATION] Execute only after recording the CP3 response.
if all_models_ready:
    # Stack training and validation rows after selection is locked: 471 + 189 = 660 cuts.
    refit = pd.concat([train, valid], ignore_index=True)
    # Extract the same 11 sensor inputs from the combined refit table.
    X_refit = refit[default_features]
    # Use the matching 660 measured wear values to learn the final model.
    y_refit = refit['wear_mean_um']
    # Extract only sensor inputs from the 285 reserved cuts.
    X_test = test[default_features]
    # Keep the measured test values separate; they are used to score predictions, not fit.
    y_test = test['wear_mean_um']
    # Create a fresh scaler rather than reusing statistics learned from only the first 50%.
    final_scaler = StandardScaler()
    # Learn means/spreads from the 660 refit rows and standardize those same rows.
    X_refit_scaled = final_scaler.fit_transform(X_refit)
    # Apply the refit statistics unchanged; do not fit a scaler on test data.
    X_test_scaled = final_scaler.transform(X_test)

### Refit B — recreate the model

Run unchanged. The matching branch keeps the selected settings;
only a Polynomial winner receives polynomial expansion.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    # Use the selected family with the validation-selected settings.
    # Only the matching branch runs; each branch creates an UNFITTED final model.
    if selected_name == 'Linear':
        final_model = LinearRegression()
    elif selected_name == 'Ridge':
        final_model = Ridge(alpha=best_alpha)
    elif selected_name == 'Polynomial':
        final_model = LinearRegression()
    # The remaining family is SVR; best_c was chosen before viewing test results.
    else:
        final_model = SVR(kernel='rbf', C=best_c, epsilon=1.0, gamma='scale')

    # Default inputs have 11 columns; only the Polynomial branch changes this.
    X_refit_final = X_refit_scaled
    X_test_final = X_test_scaled
    if selected_name == 'Polynomial':
        # Use a fresh expansion for refitting; keep the previously fixed degree of 2.
        final_polynomial = PolynomialFeatures(degree=2, include_bias=False)
        X_refit_final = final_polynomial.fit_transform(X_refit_scaled)
        # Match the refit feature columns exactly; no test targets enter preprocessing.
        X_test_final = final_polynomial.transform(X_test_scaled)


### Test — predict and score

Run unchanged. Fit on 660 rows, predict 285 test cuts, then calculate overall
and per-cutter errors using measured test wear.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    # Learn the selected model from refit inputs and targets; the test set is still excluded.
    final_model.fit(X_refit_final, y_refit)
    # Predict one wear value for each reserved cut without supplying measured test wear.
    test_pred = final_model.predict(X_test_final)
    # Convert the measured series to an array and subtract corresponding predictions.
    # Positive residual = underprediction; values remain in micrometers.
    test_residual = y_test.to_numpy() - test_pred
    # MAE averages absolute errors; RMSE gives larger errors more weight.
    final_mae = mean_absolute_error(y_test, test_pred)
    # mean_squared_error averages squared residuals; sqrt returns the result to µm.
    final_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    # R² is unitless and can be negative; it compares with the evaluated target mean.
    final_r2 = r2_score(y_test, test_pred)
    print('Selected model:', selected_name, '| Refit / test rows:', len(refit), len(test))
    print('Test MAE (µm):', round(final_mae, 2))
    print('Test RMSE (µm):', round(final_rmse, 2))
    print('Test R² (unitless):', round(final_r2, 3))
    # Start a separate table to check whether pooled errors hide differences between cutters.
    test_rows = []
    # Repeat scoring for each cutter; this loop evaluates the SAME fitted final model.
    for cutter in ['c1', 'c4', 'c6']:
        # Select this cutter’s test rows in both measured wear and predictions.
        mask = test['cutter_id'] == cutter
        # Store cutter name, MAE, and RMSE using only its 95 reserved cuts.
        test_rows.append([cutter, mean_absolute_error(y_test[mask], test_pred[mask]),
                          np.sqrt(mean_squared_error(y_test[mask], test_pred[mask]))])
    test_by_cutter = pd.DataFrame(test_rows, columns=['Cutter', 'Test MAE (µm)', 'Test RMSE (µm)'])
    display(test_by_cutter.round(2))
else:
    print('Final test waits until all four validation candidates are available.')


### Interpret the test plots

Each column is a cutter. Top: measured and predicted wear. Bottom: residuals.
Positive residuals indicate underprediction; zero means agreement.

In [ ]:
# [PROVIDED PLOT] Separate cutters so disconnected trajectories are not joined.
if all_models_ready:
    # Create 2 rows × 3 columns; shared scales make the cutters visually comparable.
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey='row')
    # enumerate supplies (0, c1), (1, c4), (2, c6): a panel column and its cutter label.
    for index, cutter in enumerate(['c1', 'c4', 'c6']):
        # Keep only the current cutter’s test rows; never join different cutters into one line.
        mask = test['cutter_id'] == cutter
        # Use physical cut numbers, not the pooled table’s row index, for the x-axis.
        cuts = test.loc[mask, 'cut_number']
        # Row 0 is the wear panel for this cutter; measured and predicted lines share the same
        # cuts.
        axes[0, index].plot(cuts, y_test[mask], label='Measured', color='#273B43')
        axes[0, index].plot(cuts, test_pred[mask], label='Predicted', color='#C08923')
        axes[0, index].set(title=cutter, ylabel='Mean flute wear (µm)')
        # Row 1 shows residuals for those same cuts; positive values mean underprediction.
        axes[1, index].plot(cuts, test_residual[mask], color='#2468A0')
        axes[1, index].axhline(0, color='black', linestyle='--')
        axes[1, index].set(xlabel='Cut number', ylabel='Residual (µm)')
    axes[0, 0].legend(fontsize=9)
    # Add one title above the entire grid, including the selected model name.
    fig.suptitle('Selected model on later cuts: ' + selected_name)
    fig.tight_layout()
    plt.show()


A validation winner may still predict later cuts poorly. Check your workflow,
then report the errors honestly; there is no required accuracy threshold.
Engineering suitability also needs an application-specific acceptable error.

### Checkpoint 4 response

Interpret the selected model's test MAE in µm and describe one cutter-specific
residual pattern or error difference. Explain why this experiment does not
demonstrate performance on an unseen cutter or prove that a sensor causes wear.

**Your response:** TODO: Write 2–3 sentences.

## Optional — all c6 cuts (ungraded)

Run unchanged to compare the final model with measured wear across c6.
**Cuts 1–220 were used for refitting; only 221–315 are held-out test data.**
Inspect where the gap grows. Do not refit, retune, or treat an all-cuts score
as test performance. No written response is required.

In [ ]:
# [OPTIONAL — PROVIDED] Reuse the final model; no fitting or tuning here.
if all_models_ready:
    # c6 is already ordered by cut number. Keep the same 11 sensor columns.
    X_c6_all = c6[default_features]
    # Apply the scaler learned from the pooled first 220 cuts of all three cutters.
    X_c6_scaled = final_scaler.transform(X_c6_all)
    # Use the 11 scaled columns unless the Polynomial branch replaces them with 77 columns.
    X_c6_final = X_c6_scaled
    # Reuse the fitted expansion only if Polynomial was selected.
    if selected_name == 'Polynomial':
        X_c6_final = final_polynomial.transform(X_c6_scaled)
    # One prediction per cut, in µm; measured wear is not passed to predict.
    c6_all_pred = final_model.predict(X_c6_final)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    # Shade the cuts used during final fitting; agreement here is in-sample performance.
    ax.axvspan(0.5, 220.5, color='#E7E7E7', label='Used in final refit (1–220)')
    # Shade the reserved test cuts separately; half-cut edges fall between observations.
    ax.axvspan(220.5, 315.5, color='#E1F0E8', label='Reserved test (221–315)')
    ax.plot(c6['cut_number'], c6['wear_mean_um'], color='#263238', linewidth=2.2, label='Measured wear')
    ax.plot(c6['cut_number'], c6_all_pred, color='#D55E00', linewidth=1.8, label='Predicted wear')
    # Draw the boundary between refit and test regions; it does not split or refit data.
    ax.axvline(220.5, color='#555555', linestyle='--', linewidth=1)
    ax.set(xlabel='Cut number', ylabel='Mean flute wear (µm)', xlim=(1, 315),
           title='c6: selected model across all cuts — ' + selected_name)
    ax.legend(loc='upper left', fontsize=10)
    fig.tight_layout()
    plt.show()
else:
    print('Complete the required model cells and final evaluation before this optional plot.')


## Optional challenge (ungraded)

Identify a train–validation gap and suggest two possible causes using existing
results. No additional code is required.

**Evaluation limit:** this Lab estimates a cut's wear from that cut's sensors
and tests later cuts of known cutters. It does not test unseen cutters,
forecast future wear, or establish causation. Distribution shift is revisited in Week 6.

## Submission and reproducibility checklist

Restart the runtime/kernel and select **Run all** after finishing your edits.
Check that all four candidates appear, a selected model is printed, and final
test results are visible. Fix code errors before exporting.

Rendered notebook checkboxes are not clickable. To record completion, edit
this Markdown cell and change `[ ]` to `[x]`.

- [ ] I entered my name and WSU AccessID.
- [ ] I completed four CP2 fit/predict edits and the four checkpoint responses.
- [ ] I explained the validation-based choice before interpreting test results.
- [ ] I restarted, ran all cells and checked required outputs and labels/units.
- [ ] I saved `Lab04_Firstname_Lastname.ipynb` and a matching PDF.
- [ ] I checked that figures and written responses are readable in the PDF.
- [ ] I submitted both files through Canvas. No separate report is required.

PDF export issues receive corrective comments, not point deductions. Required
work and submission files still need to be provided; see Canvas for active instructions.

## References

- [PHM Society: 2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/).
- Course-derived feature table and provenance: [README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md),
  [data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md).
- scikit-learn documentation: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html),
  [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html),
  [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html),
  [SVR](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html),
  [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html),
  [regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

**Dataset acknowledgment:** PHM Society 2010 Data Challenge; course-derived
scalar summaries and mean-flute-wear target. Cite the original challenge and
course feature documentation when reusing this table.